# Train CBM Concept Layer

Run this notebook for the concept bottleneck model task.

Inputs:
- `data/lyrics/final_CBM_input_data.csv` for the small concept-labeled training set
- `data/lyrics/theme_lyrics_features.csv` for manual concept labels
- `data/lyrics/master_lyrics_features.csv` for the full lyrics feature set that receives predictions

Outputs saved to Google Drive under `/content/drive/MyDrive/CLARIFY`:
- `CLARIFY/data/lyrics/concept_vectors.csv` with predicted concepts for the full lyrics feature set
- `CLARIFY/lyrics/model_outputs/concept_layer_model.pt`
- `CLARIFY/lyrics/model_outputs/concept_layer_metadata.json`

What it does:
- mounts Google Drive
- trains lyric features and BERT embeddings to predict the 15 concept scores using the labeled subset
- evaluates each concept prediction on that labeled subset
- applies the trained CBM to the full lyrics feature file
- does not train hit-score or recommender models

## 1. Setup

Imports PyTorch and sklearn. The next cell mounts Google Drive and sets `/content/drive/MyDrive/CLARIFY` as the artifact folder.

In [ ]:
# Optional in Colab if packages are missing:
# !pip install -q pandas numpy scikit-learn torch

from pathlib import Path
import json
import re
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
# Mount Google Drive and save all durable artifacts under MyDrive/CLARIFY.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/CLARIFY')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT,
    Path('/content/DS3-CLARIFY'),
    Path('/content/drive/MyDrive/DS3-CLARIFY'),
]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'lyrics' / 'final_CBM_input_data.csv').exists()),
    DRIVE_ROOT,
)

TRAINING_DATA_DIR = PROJECT_ROOT / 'data' / 'lyrics'
ARTIFACT_LYRIC_DATA_DIR = DRIVE_ROOT / 'data' / 'lyrics'
MODEL_DIR = DRIVE_ROOT / 'lyrics' / 'model_outputs'
ARTIFACT_LYRIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_INPUT_PATH = TRAINING_DATA_DIR / 'final_CBM_input_data.csv'
FULL_LYRICS_INPUT_PATH = TRAINING_DATA_DIR / 'master_lyrics_features.csv'
THEME_PATH = TRAINING_DATA_DIR / 'theme_lyrics_features.csv'
CONCEPT_VECTORS_PATH = ARTIFACT_LYRIC_DATA_DIR / 'concept_vectors.csv'
MODEL_PATH = MODEL_DIR / 'concept_layer_model.pt'
METADATA_PATH = MODEL_DIR / 'concept_layer_metadata.json'

print('Project root:', PROJECT_ROOT)
print('Drive artifact root:', DRIVE_ROOT)
print('CBM labeled input:', FINAL_INPUT_PATH, FINAL_INPUT_PATH.exists())
print('Full lyrics prediction input:', FULL_LYRICS_INPUT_PATH, FULL_LYRICS_INPUT_PATH.exists())
print('Concept labels:', THEME_PATH, THEME_PATH.exists())
print('Concept vectors save to:', CONCEPT_VECTORS_PATH)
print('CBM model artifacts save to:', MODEL_DIR)

## 2. Define Columns and Merge Training Data

Loads the lyric feature table, joins manual concept labels by song title and artist, and counts fully labeled rows.

In [ ]:
IDENTITY_COLUMNS = ["SONG_TITLE", "ARTIST_NAME", "SONG_ID"]

HANDCRAFTED_FEATURE_COLUMNS = [
    "Word_Count",
    "Unique_Word_Count",
    "Repetition_Score",
    "Average_Line_Length",
    "Vocabulary_Diversity",
    "Title_Repetition",
    "Explicit_Word_Count",
    "Sentiment_Score",
    "Positive_Score",
    "Negative_Score",
    "Emotional_Intensity",
]

CONCEPT_COLUMNS = [
    "Love",
    "Heartbreak",
    "Partying",
    "Drugs",
    "Sex",
    "Violence",
    "Self-Empowerment",
    "Success",
    "Hope",
    "Celebration",
    "Coming_Of_Age",
    "Loneliness",
    "Struggle",
    "Friendship",
    "Traveling",
]

def normalize_key(value):
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())

def embedding_sort_key(column):
    match = re.search(r"(\d+)$", column)
    return int(match.group(1)) if match else -1

def get_embedding_columns(df):
    columns = [col for col in df.columns if re.fullmatch(r"BERT_Embedding_\d+", col)]
    return sorted(columns, key=embedding_sort_key)

def require_columns(df, required, name):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

In [ ]:
final_input = pd.read_csv(FINAL_INPUT_PATH)
themes = pd.read_csv(THEME_PATH)
prediction_input = pd.read_csv(FULL_LYRICS_INPUT_PATH) if FULL_LYRICS_INPUT_PATH.exists() else final_input.copy()

def ensure_song_id(df, prefix):
    df = df.copy()
    if 'SONG_ID' not in df.columns:
        df['SONG_ID'] = [f'{prefix}_{idx:05d}' for idx in range(len(df))]
    return df

final_input = ensure_song_id(final_input, 'cbm_train')
prediction_input = ensure_song_id(prediction_input, 'lyrics_full')

require_columns(final_input, IDENTITY_COLUMNS + HANDCRAFTED_FEATURE_COLUMNS, "final_CBM_input_data.csv")
require_columns(themes, ["Song Title", "Artist"] + CONCEPT_COLUMNS, "theme_lyrics_features.csv")

embedding_columns = get_embedding_columns(final_input)
FEATURE_COLUMNS = HANDCRAFTED_FEATURE_COLUMNS + embedding_columns
require_columns(prediction_input, IDENTITY_COLUMNS + FEATURE_COLUMNS, "master_lyrics_features.csv")

final_input = final_input.copy()
themes = themes.copy()
prediction_input = prediction_input.copy()
final_input["_join_key"] = final_input["SONG_TITLE"].map(normalize_key) + "|" + final_input["ARTIST_NAME"].map(normalize_key)
themes["_join_key"] = themes["Song Title"].map(normalize_key) + "|" + themes["Artist"].map(normalize_key)

training_data = final_input.merge(
    themes[["_join_key"] + CONCEPT_COLUMNS],
    on="_join_key",
    how="left",
    validate="one_to_one",
    indicator=True,
)

unmatched = training_data["_merge"] != "both"
if unmatched.any():
    raise ValueError(training_data.loc[unmatched, IDENTITY_COLUMNS].head())

training_data = training_data[IDENTITY_COLUMNS + FEATURE_COLUMNS + CONCEPT_COLUMNS].copy()
prediction_data = prediction_input[IDENTITY_COLUMNS + FEATURE_COLUMNS].copy()

labeled_mask = training_data[CONCEPT_COLUMNS].notna().all(axis=1)
print("CBM training rows:", len(training_data))
print("Full prediction rows:", len(prediction_data))
print("Feature columns:", len(FEATURE_COLUMNS))
print("Concept columns:", len(CONCEPT_COLUMNS))
print("Fully labeled rows used for training:", int(labeled_mask.sum()))
print("This trains on labeled rows and predicts concepts for the full lyrics feature file.")
training_data.head()

## 3. Split Features and Concepts

Uses only fully labeled concept rows for training and keeps unlabeled rows for final prediction.

In [ ]:
labeled_data = training_data.loc[labeled_mask].reset_index(drop=True)

X_labeled = labeled_data[FEATURE_COLUMNS].astype(float).to_numpy()
Y_raw = labeled_data[CONCEPT_COLUMNS].astype(float).to_numpy()
X_all = prediction_data[FEATURE_COLUMNS].astype(float).to_numpy()

CONCEPT_SCALE = 10.0 if np.nanmax(Y_raw) > 1.0 else 1.0
Y = np.clip(Y_raw / CONCEPT_SCALE, 0.0, 1.0)

X_train, X_test, y_train, y_test_raw = train_test_split(
    X_labeled,
    Y_raw,
    test_size=0.2,
    random_state=SEED,
)
y_train = np.clip(y_train / CONCEPT_SCALE, 0.0, 1.0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X train:", X_train_scaled.shape)
print("X test:", X_test_scaled.shape)
print("Full prediction X:", X_all.shape)
print("Concept train:", y_train.shape)

## 4. Train Concept Predictor

Trains the CBM encoder: lyric features and BERT embeddings in, 15 concept scores out.

In [ ]:
class ConceptEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

HIDDEN_DIM = 128
DROPOUT = 0.15
BATCH_SIZE = 16
EPOCHS = 500
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

model = ConceptEncoder(
    input_dim=X_train_scaled.shape[1],
    hidden_dim=HIDDEN_DIM,
    output_dim=len(CONCEPT_COLUMNS),
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
loss_fn = nn.MSELoss()

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    if epoch % 50 == 0 or epoch == 1:
        print(f"epoch {epoch:04d} | train_mse={np.mean(losses):.5f}")

## 5. Evaluate Concept Prediction

Reports per-concept error and classification-style metrics so weak concept labels/features are visible early.

In [ ]:
def predict_numpy(model, x_scaled):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(x_scaled, dtype=torch.float32).to(DEVICE))
    return preds.cpu().numpy()

test_pred = np.clip(predict_numpy(model, X_test_scaled) * CONCEPT_SCALE, 0.0, CONCEPT_SCALE)

POSITIVE_THRESHOLD = 1.0
metric_rows = []
for idx, concept in enumerate(CONCEPT_COLUMNS):
    y_true = y_test_raw[:, idx]
    y_pred = test_pred[:, idx]

    true_binary = y_true >= POSITIVE_THRESHOLD
    pred_binary = y_pred >= POSITIVE_THRESHOLD
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_binary,
        pred_binary,
        average="binary",
        zero_division=0,
    )

    metric_rows.append({
        "Concept": concept,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
        "Accuracy": accuracy_score(true_binary, pred_binary),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    })

metrics = pd.DataFrame(metric_rows)
metrics.round(4)

## 6. Train Final Concept Encoder

Retrains on every fully labeled row, then predicts concept vectors for every song in the lyric feature table.

In [ ]:
final_scaler = StandardScaler()
X_labeled_scaled = final_scaler.fit_transform(X_labeled)
X_all_scaled = final_scaler.transform(X_all)

final_ds = TensorDataset(
    torch.tensor(X_labeled_scaled, dtype=torch.float32),
    torch.tensor(Y, dtype=torch.float32),
)
final_loader = DataLoader(final_ds, batch_size=BATCH_SIZE, shuffle=True)

final_model = ConceptEncoder(
    input_dim=X_labeled_scaled.shape[1],
    hidden_dim=HIDDEN_DIM,
    output_dim=len(CONCEPT_COLUMNS),
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

for epoch in range(1, EPOCHS + 1):
    final_model.train()
    for xb, yb in final_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(final_model(xb), yb)
        loss.backward()
        optimizer.step()

all_pred = np.clip(predict_numpy(final_model, X_all_scaled) * CONCEPT_SCALE, 0.0, CONCEPT_SCALE)
concept_vector_columns = [f"Concept_{concept}" for concept in CONCEPT_COLUMNS]
concept_vectors = pd.concat(
    [
        prediction_data[IDENTITY_COLUMNS].reset_index(drop=True),
        pd.DataFrame(all_pred, columns=concept_vector_columns),
    ],
    axis=1,
)
concept_vectors.to_csv(CONCEPT_VECTORS_PATH, index=False)
print("Saved full concept vectors:", CONCEPT_VECTORS_PATH)
print("Concept-vector rows:", len(concept_vectors))
concept_vectors.head()

## 7. Save Model Artifacts

Saves the concept vectors, trained PyTorch model, scaler values, and run metadata.

In [ ]:
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "feature_columns": FEATURE_COLUMNS,
        "concept_columns": CONCEPT_COLUMNS,
        "concept_scale": CONCEPT_SCALE,
        "scaler_mean": final_scaler.mean_,
        "scaler_scale": final_scaler.scale_,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
    },
    MODEL_PATH,
)

metadata = {
    "model_type": "pytorch_mlp_concept_layer",
    "rows": int(len(training_data)),
    "labeled_rows": int(labeled_mask.sum()),
    "full_prediction_rows": int(len(prediction_data)),
    "input_features": len(FEATURE_COLUMNS),
    "concept_outputs": len(CONCEPT_COLUMNS),
    "hidden_dim": HIDDEN_DIM,
    "dropout": DROPOUT,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "concept_scale": CONCEPT_SCALE,
    "feature_columns": FEATURE_COLUMNS,
    "concept_columns": CONCEPT_COLUMNS,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved concept vectors:", CONCEPT_VECTORS_PATH)
print("Saved model:", MODEL_PATH)
print("Saved metadata:", METADATA_PATH)